# RankCloak: Character-Level and Subword LLM Steganography for Cryptographic Artifacts

RankCloak studies LLM rank-transcoding steganography for deterministic synthetic cryptographic-artifact-like payloads. This notebook explains the research design and reads the tables produced by `scripts/run_experiment.py` or the `rankcloak` CLI.


## Research Motivation

Raw cryptographic artifacts are high-entropy strings. SHA-256 digests, random key-like values, nonces, UUID-like values, fake bearer-token-like strings, synthetic HMAC-like tags, invalid JWT-like strings, and ciphertext-like base64 blocks are difficult payloads for direct token-rank steganography because an LLM often assigns their next tokens low probability and therefore high ranks.

All examples are deterministic and synthetic. Do not use real secrets, real accounts, real credentials, real private keys, or real service tokens.


## Calgacus Background

The Calgacus method can be summarized as:

- tokenize the secret text
- compute each secret token's rank under the LLM next-token distribution
- generate cover text from a key prompt by selecting the cover token at the same rank
- decode by recomputing ranks from the same model, tokenizer, quantization, prompt, and preceding cover tokens

The Calgacus paper identifies the same limitation this repository targets: hash-like high-entropy text can produce high ranks and broken stegotext. RankCloak compares direct subword-token ranks with bounded-rank encodings that constrain ranks to small alphabets.


## Research Questions

RQ1: How does direct subword tokenization behave for cryptographic artifacts?

RQ2: Does character-level or bounded-rank encoding improve plausibility and recovery?

RQ3: Which cover disguises tolerate larger rank alphabets?

RQ4: What is the CPU cost of local experiments?


## From Smoke Prototype To Empirical Sweep

The first prototype embedded most work in this notebook. The repository now uses `scripts/run_experiment.py` as the reproducible source of truth:

```bash
python3 scripts/run_experiment.py --profile codec-only --overwrite
python3 scripts/run_experiment.py --profile smoke --overwrite
python3 scripts/run_experiment.py --profile small --overwrite
```

The notebook remains a readable research report and loads the current outputs if they exist.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
from pathlib import Path

import pandas as pd

from rankcloak.model_io import existing_llama3_model_path
from rankcloak.synthetic_payloads import generate_synthetic_payloads

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results" / "rankcloak_crypto_artifact_exploration"
FIGURES_DIR = RESULTS_DIR / "figures"

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("CPU count:", os.cpu_count())
print("Model path:", existing_llama3_model_path())
print("Results dir:", RESULTS_DIR)


## Synthetic Payloads

The payload generator uses fixed seeds and explicitly marks payloads as synthetic. These strings are safe examples for publication; they are not credentials.


In [ ]:
payloads = generate_synthetic_payloads()
payload_frame = pd.DataFrame(
    [
        {
            "payload_name": payload.name,
            "payload_kind": payload.kind,
            "byte_length": len(payload.bytes_value),
            "character_length": len(payload.text),
            "is_synthetic": payload.is_synthetic,
            "description": payload.description,
        }
        for payload in payloads
    ]
)
display(payload_frame)


## Load Experiment Tables

The helpers below load generated outputs if present. Missing files are represented as empty data frames so the notebook can run before a model-backed experiment has been executed.


In [ ]:
def load_csv(name: str) -> pd.DataFrame:
    path = RESULTS_DIR / name
    if path.exists():
        print("loaded", path)
        return pd.read_csv(path)
    print("missing", path)
    return pd.DataFrame()


def load_jsonl(name: str) -> pd.DataFrame:
    path = RESULTS_DIR / name
    if not path.exists():
        print("missing", path)
        return pd.DataFrame()
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("loaded", path)
    return pd.DataFrame(rows)


tokenization_audit = load_csv("tokenization_audit.csv")
rank_statistics = load_csv("rank_statistics.csv")
codec_roundtrip_trials = load_csv("codec_roundtrip_trials.csv")
stegotext_recovery_trials = load_csv("stegotext_recovery_trials.csv")
baseline_cover_examples = load_jsonl("baseline_cover_examples.jsonl")
cover_text_features = load_csv("cover_text_features.csv")


## Tokenization Audit


In [ ]:
display(tokenization_audit.head(20))


## Direct Subword Rank Statistics


In [ ]:
display(rank_statistics.head(20))


## Bounded-Rank Encoding

`codec_roundtrip_trials.csv` measures byte/rank codec correctness without using the generated text channel. This is intentionally separate from `stegotext_recovery_trials.csv`, which measures the full cover-text channel.


In [ ]:
display(codec_roundtrip_trials.head(20))
if not codec_roundtrip_trials.empty:
    display(codec_roundtrip_trials.groupby("alphabet_size")["rank_count"].mean().reset_index())


## Full Stegotext Recovery Trials


In [ ]:
display(stegotext_recovery_trials.head(20))


## Baseline Cover Generation


In [ ]:
display(baseline_cover_examples.head(10))


## Plausibility And Detectability Features

These lightweight features are descriptive only. The repository does not yet claim detector AUC because no classifier evaluation is implemented in this first empirical framework pass.


In [ ]:
display(cover_text_features.head(20))
if not cover_text_features.empty:
    feature_columns = [
        "whitespace_fraction",
        "punctuation_fraction",
        "digit_fraction",
        "alphabetic_fraction",
        "unique_token_fraction",
    ]
    available = [column for column in feature_columns if column in cover_text_features.columns]
    display(cover_text_features.groupby("source_type")[available].mean())


## Reproducibility Manifest


In [ ]:
manifest_path = RESULTS_DIR / "MANIFEST.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(json.dumps(manifest, indent=2))
else:
    print("Missing", manifest_path)


## Figures


In [ ]:
from IPython.display import Image, display as ipy_display

for figure_name in [
    "token_count_by_payload.png",
    "rank_summary_direct_subword.png",
    "cover_length_vs_rank_alphabet.png",
    "recovery_by_cover_prompt_and_alphabet.png",
    "cover_text_feature_comparison.png",
]:
    figure_path = FIGURES_DIR / figure_name
    if figure_path.exists():
        print(figure_name)
        ipy_display(Image(filename=str(figure_path)))
    else:
        print("missing", figure_path)


## Limitations

- RankCloak is not encryption, key exchange, authentication, or cryptographic security.
- Exact recovery requires the same model, tokenizer, quantization, deterministic rank ordering, cover prompt, and unmodified generated text.
- Public text channels must preserve generated text exactly.
- All examples are deterministic and synthetic.
- High-entropy payloads are harder than natural-language payloads.
- Feature extraction is not detector AUC.


## Next Experiments

- Run the `small` profile on CPU when time is available.
- Add a larger cover genre sweep.
- Compare Phi, Llama, Mistral, Qwen, and Gemma.
- Add detector AUC experiments after feature extraction stabilizes.
- Implement distribution-matched rank coding.
- Test robustness to copy/paste edits, punctuation normalization, and paraphrase.
- Run a human or LLM plausibility study.
